# Paligemma 3b COCO Pruned GA P30 Mr002

This notebook was reorganized for the GitHub reproducibility package.
Original file: `GA-I_P30_MR0.02/PaliGemma_Pruned[1,5,3,2]#L01f44c.ipynb_`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


# **PaliGemma Prune Çalışmaları_2**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y default-jdk -q
!pip install pymoo pycocotools pycocoevalcap -q

In [ ]:
# Diske kopyalama

from google.colab import drive
import os, json, shutil, time

drive.mount('/content/drive')

COCO_DST    = "/content/images/coco_val2014"
NOCAPS_DST  = "/content/images/nocaps_val"
os.makedirs(COCO_DST, exist_ok=True)
os.makedirs(NOCAPS_DST, exist_ok=True)

# ── Sadece test setindeki 5000 COCO görüntüsünü kopyala ──
COCO_SRC  = "/content/drive/MyDrive/datasets/coco2014/val2014"
TEST_JSON = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"

with open(TEST_JSON) as f:
    test_images = json.load(f)

print(f"COCO test görüntüleri kopyalanıyor ({len(test_images)} adet)...")
t0 = time.time()
for item in test_images:
    fname = item["image"].split("/")[-1]
    src = os.path.join(COCO_SRC, fname)
    dst = os.path.join(COCO_DST, fname)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
print(f"✅ COCO kopyalandı — {time.time()-t0:.0f} sn")

# ── NoCaps 4500 görüntüyü kopyala (zaten küçük) ──
NOCAPS_SRC     = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"
GT_JSON_NOCAPS = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"

with open(GT_JSON_NOCAPS) as f:
    nocaps_data = json.load(f)
nocaps_images = nocaps_data["images"]

print(f"NoCaps görüntüleri kopyalanıyor ({len(nocaps_images)} adet)...")
t0 = time.time()
for item in nocaps_images:
    fname = item["file_name"]
    src = os.path.join(NOCAPS_SRC, fname)
    dst = os.path.join(NOCAPS_DST, fname)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
print(f"✅ NoCaps kopyalandı — {time.time()-t0:.0f} sn")

In [ ]:
# Import + base model yükle

import os, json, torch, copy, time
from PIL import Image
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from huggingface_hub import login

login()  # Enter your Hugging Face token interactively; never commit tokens.  # Enter your Hugging Face token interactively; never commit tokens.

MODEL_ID    = "google/paligemma-3b-ft-cococap-448"
DEVICE      = "cuda"
BATCH_SIZE  = 64
PARETO_JSON = "/content/drive/MyDrive/ga_results/paligemma/pareto_solutions.json"
SELECTED    = [1, 5, 3, 2]
label_map   = {1: "low", 5: "mid", 3: "high", 2: "extra_high"}

with open(PARETO_JSON) as f:
    pareto_data = json.load(f)
selected_solutions = [s for s in pareto_data["solutions"] if s["index"] in SELECTED]
selected_solutions.sort(key=lambda x: x["param_drop_rate"])

print("Seçilen çözümler:")
for s in selected_solutions:
    print(f"  index:{s['index']} | %{s['param_drop_rate']*100:.1f} | proxy CIDEr:{s['cider_score']:.4f}")

print("\nBase model yükleniyor...")
processor = PaliGemmaProcessor.from_pretrained(MODEL_ID)
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    attn_implementation="eager"
).to(DEVICE)
base_model.eval()
total_params = sum(p.numel() for p in base_model.parameters())
print(f"✅ Base model hazır | Parametre: {total_params:,}")


def prune_model(base_model, chromosome):
    model = copy.deepcopy(base_model)
    layers = model.model.language_model.layers
    keep_indices = [i for i, keep in enumerate(chromosome) if keep]
    new_layers = torch.nn.ModuleList([layers[i] for i in keep_indices])
    for layer in new_layers:
        for param in layer.parameters():
            if not param.is_contiguous():
                param.data = param.data.contiguous()
    model.model.language_model.layers = new_layers
    return model


def run_inference_paligemma(model, processor, images, img_dir, id_fn, device, batch_size=64):
    """PaliGemma decoder-only — TRIM FIX VAR"""
    results = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        pil_imgs, img_ids = [], []
        for img_info in batch:
            fname, img_id = id_fn(img_info)
            pil_imgs.append(Image.open(os.path.join(img_dir, fname)).convert("RGB"))
            img_ids.append(img_id)

        inputs = processor(
            images=pil_imgs,
            text=["caption en"] * len(pil_imgs),
            return_tensors="pt"
        ).to(device, torch.float16)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=50,
                num_beams=1,
                do_sample=False
            )

        # ── TRIM FIX: decoder-only, input token'larını kırp ──
        generated_ids = out[:, inputs.input_ids.shape[1]:]
        captions = processor.batch_decode(generated_ids, skip_special_tokens=True)

        for img_id, cap in zip(img_ids, captions):
            results.append({"image_id": img_id, "caption": cap.strip()})

        if (i // batch_size) % 50 == 0:
            print(f"  {i+len(batch)}/{len(images)}")
            print(f"  Ham çıktı   : {processor.decode(out[0], skip_special_tokens=False)[:80]}")
            print(f"  Trim sonrası: {captions[0]}")

    return results


def compute_metrics(results, gt_json, save_dir):
    res_path = os.path.join(save_dir, "_tmp_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f)
    coco_gt   = COCO(gt_json)
    coco_res  = coco_gt.loadRes(res_path)
    evaluator = COCOEvalCap(coco_gt, coco_res)
    try:
        evaluator.evaluate()
    except Exception as e:
        print(f"  ⚠️ SPICE hatası (görmezden gelindi): {e}")
    return {k: v for k, v in evaluator.eval.items() if k in ["CIDEr", "Bleu_4"]}

print("✅ Tüm fonksiyonlar hazır")

***Pareto çözümü olarak ilk 4 indeksi alıyoruz 1 - 5 - 3 - 2***

In [ ]:
# COCO Final Evaluation

IMG_DIR   = COCO_DST
GT_JSON   = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json"
TEST_JSON = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"
SAVE_DIR  = "/content/drive/MyDrive/ga_results/paligemma/final_eval_2"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(TEST_JSON) as f:
    test_images = json.load(f)
print(f"COCO test seti: {len(test_images)} görüntü")

def coco_id_fn(img_info):
    fname  = img_info["image"].split("/")[-1]
    img_id = int(fname.split("_")[-1].split(".")[0])
    return fname, img_id

summary = []
t_total = time.time()

for sol in selected_solutions:
    label = label_map[sol["index"]]
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş")

    model = prune_model(base_model, sol["chromosome"])
    n_kept = sum(sol["chromosome"])
    print(f"  {len(sol['chromosome']) - n_kept} blok silindi, {n_kept} kaldı")

    t0 = time.time()
    results = run_inference_paligemma(
        model, processor, test_images,
        IMG_DIR, coco_id_fn, DEVICE, BATCH_SIZE
    )
    metrics = compute_metrics(results, GT_JSON, SAVE_DIR)

    cider = metrics.get("CIDEr", 0)
    bleu4 = metrics.get("Bleu_4", 0)
    print(f"  CIDEr : {cider:.4f}")
    print(f"  BLEU-4: {bleu4:.4f}")
    print(f"  Süre  : {(time.time()-t0)/60:.1f} dk")

    entry = {
        "label": label,
        "index": sol["index"],
        "param_drop_rate": sol["param_drop_rate"],
        "proxy_cider": sol["cider_score"],
        "final_cider": cider,
        "final_bleu4": bleu4,
        "chromosome": sol["chromosome"]
    }
    summary.append(entry)

    out_path = os.path.join(SAVE_DIR, f"paligemma_{label}_eval_2.json")
    with open(out_path, "w") as f:
        json.dump(entry, f, indent=2)
    print(f"  ✅ Kaydedildi: {out_path}")

    del model
    torch.cuda.empty_cache()

summary_path = os.path.join(SAVE_DIR, "paligemma_final_summary_2.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*50}")
print(f"✅ COCO tamamlandı — toplam süre: {(time.time()-t_total)/60:.1f} dk")
print(f"\nÖZET:")
print(f"{'Label':<12} {'Param Düşüş':>12}  {'Proxy CIDEr':>12}  {'Final CIDEr':>12}  {'BLEU-4':>8}")
print("-" * 62)
for s in summary:
    print(f"{s['label']:<12} %{s['param_drop_rate']*100:>10.1f}  "
          f"{s['proxy_cider']:>12.4f}  {s['final_cider']:>12.4f}  {s['final_bleu4']:>8.4f}")

In [ ]:
# NoCaps Final Evaluation


IMG_DIR_NOCAPS  = NOCAPS_DST
GT_JSON_NOCAPS  = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
SAVE_DIR_NOCAPS = "/content/drive/MyDrive/ga_results/paligemma/final_eval_nocaps_2"
os.makedirs(SAVE_DIR_NOCAPS, exist_ok=True)

with open(GT_JSON_NOCAPS) as f:
    nocaps_data = json.load(f)
nocaps_images = nocaps_data["images"]
print(f"NoCaps val seti: {len(nocaps_images)} görüntü")

def nocaps_id_fn(img_info):
    return img_info["file_name"], img_info["id"]

summary_nocaps = []
t_total = time.time()

for sol in selected_solutions:
    label = label_map[sol["index"]]
    print(f"\n{'='*50}")
    print(f"[{label.upper()}] index:{sol['index']} | %{sol['param_drop_rate']*100:.1f} düşüş")

    model = prune_model(base_model, sol["chromosome"])
    n_kept = sum(sol["chromosome"])
    print(f"  {len(sol['chromosome']) - n_kept} blok silindi, {n_kept} kaldı")

    t0 = time.time()
    results = run_inference_paligemma(
        model, processor, nocaps_images,
        IMG_DIR_NOCAPS, nocaps_id_fn, DEVICE, BATCH_SIZE
    )
    metrics = compute_metrics(results, GT_JSON_NOCAPS, SAVE_DIR_NOCAPS)

    cider = metrics.get("CIDEr", 0)
    bleu4 = metrics.get("Bleu_4", 0)
    print(f"  CIDEr : {cider:.4f}")
    print(f"  BLEU-4: {bleu4:.4f}")
    print(f"  Süre  : {(time.time()-t0)/60:.1f} dk")

    entry = {
        "label": label,
        "index": sol["index"],
        "param_drop_rate": sol["param_drop_rate"],
        "proxy_cider": sol["cider_score"],
        "final_cider": cider,
        "final_bleu4": bleu4,
        "chromosome": sol["chromosome"]
    }
    summary_nocaps.append(entry)

    out_path = os.path.join(SAVE_DIR_NOCAPS, f"paligemma_nocaps_{label}_eval_2.json")
    with open(out_path, "w") as f:
        json.dump(entry, f, indent=2)
    print(f"  ✅ Kaydedildi: {out_path}")

    del model
    torch.cuda.empty_cache()

summary_path = os.path.join(SAVE_DIR_NOCAPS, "paligemma_nocaps_summary_2.json")
with open(summary_path, "w") as f:
    json.dump(summary_nocaps, f, indent=2)

print(f"\n{'='*50}")
print(f"✅ NoCaps tamamlandı — toplam süre: {(time.time()-t_total)/60:.1f} dk")
print(f"\nÖZET:")
print(f"{'Label':<12} {'Param Düşüş':>12}  {'Proxy CIDEr':>12}  {'Final CIDEr':>12}  {'BLEU-4':>8}")
print("-" * 62)
for s in summary_nocaps:
    print(f"{s['label']:<12} %{s['param_drop_rate']*100:>10.1f}  "
          f"{s['proxy_cider']:>12.4f}  {s['final_cider']:>12.4f}  {s['final_bleu4']:>8.4f}")